# ODE-Multiomics Benchmark: Baseline Run on Colab GPU

This notebook runs the full baseline experiment (500 patients, 5 replicates, 500 UDE epochs) on Colab's free T4 GPU (~30–90 min runtime), saves results to Google Drive, and downloads them locally.

**Before running:**
1. Set runtime to GPU: `Runtime > Change runtime type > T4 GPU`
2. Choose an option below (GitHub clone OR zip upload)

## Cell 1: Setup & Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
OUTPUT_DIR = '/content/drive/MyDrive/ode-multiomics-results'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Results will be saved to: {OUTPUT_DIR}')

## Cell 2: Get Code (Pick ONE option below)

### Option A: Clone from GitHub (Recommended)

In [ ]:
!git clone https://github.com/Model-Stewardship/ode-multiomics-benchmark.git
%cd ode-multiomics-benchmark
!git log --oneline -3

### Option B: Upload from Google Drive (Fallback)
If GitHub option fails, uncomment below and make sure you've uploaded `ode-multiomics-benchmark.zip` to your Drive's root.

In [ ]:
# !cp '/content/drive/MyDrive/ode-multiomics-benchmark.zip' .
# !unzip -q ode-multiomics-benchmark.zip
# %cd ode-multiomics-benchmark
# !ls -la

## Cell 3: Install Dependencies

In [ ]:
# torch, numpy, scipy, scikit-learn, pandas already in Colab base image
!pip install -q torchdiffeq pysindy cloudpickle tqdm
print('Dependencies installed')

## Cell 4: Verify GPU & Environment

In [ ]:
import torch
import sys

cuda_available = torch.cuda.is_available()
print(f'CUDA available: {cuda_available}')
if cuda_available:
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  WARNING: No GPU detected. Did you set runtime to T4?')

print(f'\nPython: {sys.version}')
print(f'PyTorch: {torch.__version__}')

## Cell 5: Run Baseline Experiment

In [ ]:
# This will take 30–90 minutes depending on GPU performance
# Progress: MOTIF calibration per replicate, then UDE training loss curves
import subprocess
result = subprocess.run(
    ['python', '-m', 'src.run_experiment', '--config', 'experiments/config_baseline.yaml'],
    capture_output=False,
    text=True
)
exit_code = result.returncode
print(f'\nExperiment exited with code: {exit_code}')

## Cell 6: Copy Results to Google Drive

In [ ]:
import shutil
import glob
import os

results_copied = []
for run_dir in glob.glob('results/baseline_*'):
    basename = os.path.basename(run_dir)
    dest = os.path.join(OUTPUT_DIR, basename)
    print(f'Copying {basename}...')
    shutil.copytree(run_dir, dest, dirs_exist_ok=True)
    results_copied.append(basename)
    print(f'  ✓ Saved to Google Drive')

if results_copied:
    print(f'\n✓ Done! {len(results_copied)} result folder(s) saved to Google Drive.')
    print(f'Location: ode-multiomics-results/{results_copied[0]}')
    print('\nNext: Download the results folder to your machine for local analysis.')
else:
    print('No baseline results found. Check Cell 5 for errors.')

## Cell 7 (Optional): Quick Quality Check
Load and display summary metrics from the run.

In [ ]:
import json
import glob

# Find the most recent baseline run's metrics
metrics_files = glob.glob('results/baseline_*/metrics.json')
if metrics_files:
    metrics_files.sort()
    with open(metrics_files[-1]) as f:
        metrics = json.load(f)
    
    print('Baseline Run Summary:')
    print(f"  Timestamp: {metrics.get('timestamp', 'N/A')}")
    print(f"  Replicates: {metrics.get('n_replicates', 'N/A')}")
    print(f"  Patients per replicate: {metrics.get('n_patients', 'N/A')}")
    if 'motif' in metrics:
        print(f"  MOTIF Mean R²: {metrics['motif'].get('mean_r2', 'N/A'):.4f}")
    if 'ude' in metrics:
        print(f"  UDE Final Loss: {metrics['ude'].get('mean_loss', 'N/A')}")
else:
    print('No metrics found. Make sure Cell 5 completed successfully.')